In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path

NOTEBOOK_START_MONOTONIC = time.perf_counter()
print("L4_DUAL_TTT_DFS1024_NOTEBOOK_BENCH_V1", flush=True)
print("RERUN_ENABLED = FALSE", flush=True)
DATASET = Path("/kaggle/input/datasets/jimmy5566/arc2-l4-dual-ttt-dfs1024-bench-v1-source")
SOURCE = DATASET / "ARC2"
CHALLENGE = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json")
MODEL = Path("/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1")
RUNNER = SOURCE / "scripts/run_l4_dual_ttt_dfs1024_benchmark.py"
NATIVE_CONFIG = SOURCE / "configs/nvarc_native_846d0198"
CONFIG = DATASET / "benchmark_config_resolved.json"
COHORT = DATASET / "BENCHMARK_TASK_IDS.json"
IDENTITY = DATASET / "SOURCE_IDENTITY.json"
OUTPUT = Path("/kaggle/working/analysis/l4_dual_ttt_dfs1024_bench_v1")
required = [SOURCE, CHALLENGE, MODEL, RUNNER, NATIVE_CONFIG, CONFIG, COHORT, IDENTITY]
missing = [str(path) for path in required if not path.exists()]
if missing: raise RuntimeError(f"benchmark inputs missing: {missing}")
if OUTPUT.exists() and any(OUTPUT.iterdir()): raise RuntimeError("RERUN_ENABLED=FALSE: benchmark output already exists")
if os.environ.get("KAGGLE_KERNEL_INTERNET_ENABLED", "").strip().lower() in {"1", "true", "yes"}: raise RuntimeError("Internet must be disabled")
gpus = subprocess.check_output(["nvidia-smi", "-L"], text=True).splitlines()
if len(gpus) != 4 or any("NVIDIA L4" not in row for row in gpus): raise RuntimeError(f"requires exactly four NVIDIA L4 GPUs: {gpus}")
identity = json.loads(IDENTITY.read_text(encoding="utf-8"))
env_init_s = time.perf_counter() - NOTEBOOK_START_MONOTONIC
command = [sys.executable, str(RUNNER), "--challenge", str(CHALLENGE), "--model-path", str(MODEL), "--native-config-dir", str(NATIVE_CONFIG), "--config", str(CONFIG), "--cohort", str(COHORT), "--output", str(OUTPUT), "--source-branch", identity["source_branch"], "--source-git-status", identity["source_git_status"], "--notebook-start-monotonic", str(NOTEBOOK_START_MONOTONIC), "--env-init-seconds", str(env_init_s)]
print(json.dumps({"event": "BENCHMARK_NOTEBOOK_PREFLIGHT_PASS", "dataset": identity, "gpus": gpus, "internet": False, "rerun": False}, sort_keys=True), flush=True)
subprocess.run(command, check=True, env={**os.environ, "TRITON_PTXAS_PATH": "/usr/local/cuda-12.5/bin/ptxas", "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "TOKENIZERS_PARALLELISM": "false"})
required_outputs = ["benchmark_provenance.json", "BENCHMARK_TASK_IDS.json", "BENCHMARK_COHORT_SHA256.txt", "task_timing.csv", "cell_dfs_metrics.csv", "gpu_telemetry_1s.csv", "gpu_summary.csv", "system_telemetry_1s.csv", "worker_summary.csv", "throughput_summary.json", "runtime_projection.json", "failures.jsonl", "startup_summary.json", "L4_DUAL_TTT_DFS1024_BENCH_REPORT.md", "DECISION.json"]
missing_outputs = [name for name in required_outputs if not (OUTPUT / name).is_file()]
if missing_outputs: raise RuntimeError(f"benchmark outputs missing: {missing_outputs}")
decision = json.loads((OUTPUT / "DECISION.json").read_text(encoding="utf-8"))
if decision.get("status") != "PASS" or decision.get("gold_accessed") is not False or decision.get("rerun_enabled") is not False: raise RuntimeError(f"benchmark decision failed: {decision}")
print(json.dumps({"event": "BENCHMARK_NOTEBOOK_COMPLETE", "output": str(OUTPUT), "status": decision["status"], "rerun": False}, sort_keys=True), flush=True)
